In [ ]:
import pickle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
import xgboost as xgb


pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('telcoIbm.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors='coerce').fillna(0)
df.drop(columns = ['customerID'], inplace=True)
df['Churn'] = df['Churn'].map({'Yes':1, 'No': 0})
df[["TotalCharges","Churn"]].info()

In [ ]:
df["Charge_Ratio"] = df["MonthlyCharges"]/(df["TotalCharges"]+1)
df[["MonthlyCharges", "TotalCharges","tenure", "Charge_Ratio"]].head()

In [ ]:
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
df_encoded = pd.get_dummies(df, columns = categorical_cols, drop_first=True)
print(f"New encoded shape: {df_encoded}")
df_encoded.head()

In [ ]:
X = df_encoded.drop(columns=["Churn"])
y = df_encoded["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scale_pos_weight = (y_train == 0).sum()/ (y_train == 1).sum()

print(f"Training rows: {X_train.shape[0]} | Testing rows: {X_test.shape[0]}")
print(f"Calculated scale_pos_weight: {scale_pos_weight:.2f}")


In [ ]:
model = xgb.XGBClassifier(
    n_estimators = 150,
    max_depth = 4,
    learning_rate = 0.05,
    scale_pos_weight = scale_pos_weight,
    subsample = 0.8,
    random_state = 42,
    eval_metric = "logloss",
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["Retained", "Churn"]))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

In [ ]:
with open("modle.pkl", "wb") as f:
    pickle.dump(model,f)
with open("model_columns.pkl", "wb") as f:
    pickle.dump(X.columns.tolist(),f)

print("Saved 'model.pkl' and 'model_columns.pkl'successfully." )